# SE4050 Deep Learning Assignment
## Notebook 4: VGG16 Transfer Learning
### Brain Tumor MRI Classification — Model 2 of 4

**Architecture**: VGG16 pretrained on ImageNet-1K with a custom classification head  
**Training strategy**: Two-phase approach — feature extraction (frozen backbone) followed by partial fine-tuning  
**Justification**: VGG16 (Simonyan and Zisserman, 2015) comprises 13 convolutional layers
with uniform 3x3 filters, organised into five progressive blocks. Its depth enables hierarchical
feature learning: early layers capture low-level textures (edges, gradients) while deeper layers
capture high-level semantic structures. Pre-trained ImageNet weights provide rich feature
representations that transfer effectively from natural images to MRI data, particularly for
detecting structural anomalies at multiple scales.

**Prerequisite**: Execute `02_Preprocessing.ipynb` before this notebook.

## Section 0: Environment Setup

In [ ]:
# ---------------------------------------------------------------------------
# Set the Keras backend to PyTorch before any deep learning imports.
# This enables CUDA GPU acceleration on Windows where TensorFlow GPU
# is not supported in versions >= 2.11.
# ---------------------------------------------------------------------------
import os
os.environ['KERAS_BACKEND'] = 'torch'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import json, time, random, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch and torchvision
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# Evaluation metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    accuracy_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Reproducibility seed — must match the seed used in preprocessing
# ---------------------------------------------------------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU model      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM available : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DATA_DIR  = Path('preprocessed_data')
MODEL_DIR = Path('saved_models') / 'VGG16'
RES_DIR   = Path('results')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Hyperparameters — separate learning rates for each training phase.
# Phase 1 uses a higher LR because only the small custom head is updated.
# Phase 2 uses a much lower LR to avoid disrupting the pre-trained weights.
# ---------------------------------------------------------------------------
BATCH_SIZE    = 32
PHASE1_EPOCHS = 15    # Feature extraction: frozen backbone, train head only
PHASE2_EPOCHS = 35    # Fine-tuning: unfreeze upper backbone layers
LR_PHASE1     = 1e-4  # Learning rate for Phase 1
LR_PHASE2     = 1e-5  # Lower LR for Phase 2 to prevent catastrophic forgetting
MODEL_NAME    = 'VGG16'

print(f'Phase 1 (feature extraction) : up to {PHASE1_EPOCHS} epochs @ LR={LR_PHASE1}')
print(f'Phase 2 (fine-tuning)        : up to {PHASE2_EPOCHS} epochs @ LR={LR_PHASE2}')

## Section 1: Load Preprocessed Data with VGG16-Specific Normalisation

VGG16 was pre-trained on ImageNet with inputs normalised using the dataset's channel-wise
mean and standard deviation. Applying the same statistics to our MRI inputs aligns the
distribution of the input features with what the pre-trained filters expect, which is
critical for effective transfer learning.

In [ ]:
# Load the preprocessed arrays from 02_Preprocessing.ipynb
X_train = np.load(DATA_DIR / 'X_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
X_val   = np.load(DATA_DIR / 'X_val.npy')
y_val   = np.load(DATA_DIR / 'y_val.npy')
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy')

class_weights_arr    = np.load(DATA_DIR / 'class_weights.npy')
CLASS_NAMES          = np.load(DATA_DIR / 'class_names.npy', allow_pickle=True).tolist()
NUM_CLASSES          = len(CLASS_NAMES)
CLASS_COLORS         = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
class_weights_tensor = torch.FloatTensor(class_weights_arr).to(DEVICE)

# ImageNet channel-wise mean and standard deviation used for VGG16 pre-training.
# Applying these ensures that our MRI input distribution matches the training distribution
# of the pre-trained model, enabling effective feature transfer.
VGG_MEAN = [0.485, 0.456, 0.406]
VGG_STD  = [0.229, 0.224, 0.225]

print(f'Arrays loaded:')
print(f'  X_train : {X_train.shape}  |  X_val : {X_val.shape}  |  X_test : {X_test.shape}')
print(f'  Classes : {CLASS_NAMES}')
print(f'\nVGG16 ImageNet normalisation statistics:')
print(f'  Mean : {VGG_MEAN}')
print(f'  Std  : {VGG_STD}')

## Section 2: Dataset Class with VGG16 Normalisation

In [ ]:
class MRIDatasetVGG(Dataset):
    """
    PyTorch Dataset with VGG16-specific preprocessing applied via torchvision transforms.

    Training images receive stochastic augmentation followed by ImageNet normalisation.
    Validation and test images receive only normalisation (no augmentation).

    Note: The base [0, 1] arrays from preprocessing are scaled back to [0, 255] uint8
    before passing to T.ToPILImage(), which requires uint8 input.
    The final T.Normalize() step then applies ImageNet mean subtraction.
    """

    # Training pipeline: augmentation + ImageNet normalisation
    TRAIN_TRANSFORMS = T.Compose([
        T.ToPILImage(),
        T.RandomRotation(40),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.3),
        T.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=20, scale=(0.8, 1.2)),
        T.ColorJitter(brightness=0.2, contrast=0.15),
        T.ToTensor(),                         # -> Tensor (C, H, W) in [0, 1]
        T.Normalize(mean=VGG_MEAN, std=VGG_STD),  # ImageNet normalisation
    ])

    # Evaluation pipeline: only normalisation (no randomness)
    EVAL_TRANSFORMS = T.Compose([
        T.ToPILImage(),
        T.ToTensor(),
        T.Normalize(mean=VGG_MEAN, std=VGG_STD),
    ])

    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        self.X         = (X * 255).astype(np.uint8)
        self.y         = y.astype(np.int64)
        self.transform = self.TRAIN_TRANSFORMS if augment else self.EVAL_TRANSFORMS

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return self.transform(self.X[idx]), self.y[idx]


g = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = DataLoader(MRIDatasetVGG(X_train, y_train, augment=True),
                          BATCH_SIZE, shuffle=True,  num_workers=0, generator=g, pin_memory=True)
val_loader   = DataLoader(MRIDatasetVGG(X_val,   y_val,   augment=False),
                          BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(MRIDatasetVGG(X_test,  y_test,  augment=False),
                          BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print('DataLoaders created:')
print(f'  Training   : {len(train_loader)} batches')
print(f'  Validation : {len(val_loader)} batches')
print(f'  Test       : {len(test_loader)} batches')

## Section 3: VGG16 Model Architecture

### Design Rationale

The original VGG16 classification head (three dense layers of 4096, 4096, and 1000 neurons)
is replaced with a custom head adapted for 4-class MRI classification. Key design decisions:

1. **Global Average Pooling** replaces the original flatten operation, reducing parameters
   from 25M (head only) to 0.4M and providing implicit spatial regularisation.
2. **Batch Normalisation** before ReLU stabilises gradient flow through the custom head.
3. **Two-phase training** (freeze -> unfreeze) prevents catastrophic forgetting:
   - Phase 1 trains only the custom head, establishing good classifier weights first.
   - Phase 2 fine-tunes the upper backbone blocks (block4 and block5) with a very
     low learning rate, allowing gradual adaptation to MRI-specific features.

In [ ]:
class VGG16TransferModel(nn.Module):
    """
    VGG16 backbone with a custom classification head for 4-class MRI classification.

    Backbone: VGG16 feature extractor (13 conv layers, 5 max-pool layers)
    Pooling : AdaptiveAvgPool2d(1) — Global Average Pooling
    Head    : Linear(512->512) -> BN -> ReLU -> Dropout(0.5)
              Linear(512->256) -> ReLU -> Dropout(0.4)
              Linear(256->4)

    Parameters
    ----------
    num_classes : number of output classes
    freeze_base : if True, freeze all backbone parameters in Phase 1
    """

    def __init__(self, num_classes: int = 4, freeze_base: bool = True):
        super().__init__()

        # Load VGG16 with official ImageNet-1K weights
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        # Extract only the convolutional feature extractor; discard VGG's FC head
        self.features       = vgg.features
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

        # Freeze all backbone parameters during Phase 1
        if freeze_base:
            for param in self.features.parameters():
                param.requires_grad = False

        # Custom classification head — Xavier initialisation for stable gradients
        # VGG16 backbone outputs 512 channels after the final conv block
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),  # Raw logits
        )

        # Initialise custom head weights for stable early training
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)         # Convolutional feature extraction
        x = self.global_avg_pool(x)  # Collapse spatial dimensions to 1x1
        return self.classifier(x)    # Classification decision

    def unfreeze_block4_and_block5(self):
        """
        Selectively unfreeze VGG16 block4 and block5 for fine-tuning.

        VGG16 layer indices in self.features:
            Block 1: 0-4   (conv-conv-maxpool)
            Block 2: 5-9   (conv-conv-maxpool)
            Block 3: 10-17 (conv-conv-conv-maxpool)
            Block 4: 17-24 (conv-conv-conv-maxpool)  <-- unfrozen
            Block 5: 24-30 (conv-conv-conv-maxpool)  <-- unfrozen
        Blocks 1-3 remain frozen to preserve low-level edge/texture features
        that are generally transferable across image domains.
        """
        unfrozen_params = 0
        for name, param in self.features.named_parameters():
            # Layer index is the numeric prefix of the parameter name (e.g. "17.weight")
            layer_idx = int(name.split('.')[0]) if name.split('.')[0].isdigit() else 0
            if layer_idx >= 17:  # Block 4 starts at index 17
                param.requires_grad = True
                unfrozen_params += param.numel()
        print(f'Unfrozen block4 and block5: {unfrozen_params:,} parameters enabled for training.')


# Instantiate model
model = VGG16TransferModel(num_classes=NUM_CLASSES, freeze_base=True).to(DEVICE)

frozen_p    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p     = frozen_p + trainable_p
print(f'Total parameters             : {total_p:,}')
print(f'Trainable (Phase 1, head)    : {trainable_p:,}')
print(f'Frozen (Phase 1, backbone)   : {frozen_p:,}')

# Forward-pass dimension check
with torch.no_grad():
    dummy_out = model(torch.randn(2, 3, 224, 224).to(DEVICE))
print(f'Forward pass check           : (2, 3, 224, 224) -> {tuple(dummy_out.shape)}')

## Section 4: Training Utilities

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """One training epoch: forward pass, loss computation, backpropagation, weight update."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        total      += len(y_b)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model on a loader without computing gradients."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits    = model(X_b)
        loss      = criterion(logits, y_b)
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        total      += len(y_b)
    return total_loss / total, correct / total


class EarlyStopping:
    """Stop training when validation loss ceases to improve for `patience` epochs."""
    def __init__(self, patience=12, min_delta=1e-4):
        self.patience, self.min_delta = patience, min_delta
        self.counter = 0; self.best_loss = float('inf'); self.should_stop = False
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0; return True
        self.counter += 1
        if self.counter >= self.patience: self.should_stop = True
        return False


def run_training_phase(model, train_loader, val_loader, criterion, optimizer,
                       scheduler, max_epochs, model_path, phase_label, history=None):
    """
    Execute a complete training phase (either Phase 1 or Phase 2).

    This function is shared between both phases to avoid code duplication.
    For Phase 2, the existing `history` dict from Phase 1 is passed in so
    that the training curves span the full training duration.

    Parameters
    ----------
    history : existing history dict (pass None for Phase 1, Phase 1's history for Phase 2)
    """
    hist = history or {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    es   = EarlyStopping(patience=10)
    best = float('inf')

    print(f'\n{"="*68}')
    print(f'  {phase_label}')
    print(f'{"="*68}')
    print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>10} | {"Val Loss":>10} | {"Val Acc":>10}')
    print('-' * 58)

    t0 = time.time()
    for ep in range(1, max_epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        va_loss, va_acc = evaluate(model,    val_loader,   criterion, DEVICE)
        scheduler.step(va_loss)
        improved = es(va_loss)

        hist['train_loss'].append(tr_loss); hist['val_loss'].append(va_loss)
        hist['train_acc'].append(tr_acc);   hist['val_acc'].append(va_acc)

        marker = ' <-- best' if improved else ''
        if improved:
            torch.save(model.state_dict(), model_path)

        print(f'{ep:>6} | {tr_loss:>10.4f} | {tr_acc:>9.2%} | {va_loss:>10.4f} | {va_acc:>9.2%}{marker}')

        if es.should_stop:
            print(f'Early stopping at epoch {ep}.')
            break

    print(f'Phase duration : {(time.time()-t0)/60:.1f} min  |  Best val loss : {es.best_loss:.4f}')
    return hist


print('Training utilities defined.')

## Section 5: Phase 1 — Feature Extraction (Frozen Backbone)

In [ ]:
# Loss: CrossEntropyLoss with class weights to handle dataset imbalance
criterion   = nn.CrossEntropyLoss(weight=class_weights_tensor)
best_path   = MODEL_DIR / 'vgg16_best.pt'

# Phase 1 optimiser: only update parameters that require gradients (the custom head)
opt_p1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1, weight_decay=1e-4
)
sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_p1, mode='min', factor=0.5, patience=5, min_lr=1e-7
)

history = run_training_phase(
    model, train_loader, val_loader, criterion, opt_p1, sch_p1,
    PHASE1_EPOCHS, best_path,
    'PHASE 1 -- Feature Extraction (backbone frozen, head trained only)'
)

## Section 6: Phase 2 — Fine-Tuning (Partial Backbone Unfreeze)

After Phase 1 has trained the classification head to a reasonable state,
Phase 2 unfreezes the upper backbone layers (blocks 4 and 5) so that
higher-level convolutional filters can adapt to MRI-specific patterns.
The backbone is loaded from the Phase 1 checkpoint before unfreezing
to ensure training begins from the best validated state.

In [ ]:
# Reload the best Phase 1 checkpoint before modifying the architecture
model.load_state_dict(torch.load(best_path, map_location=DEVICE))

# Unfreeze block4 and block5 of the VGG16 backbone
model.unfreeze_block4_and_block5()

trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 trainable parameters : {trainable_p2:,}')

# Phase 2 optimiser: uses a 10x lower learning rate than Phase 1
# to avoid large weight updates that would overwrite useful ImageNet features
opt_p2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE2, weight_decay=1e-4
)
sch_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_p2, mode='min', factor=0.5, patience=5, min_lr=1e-8
)

history = run_training_phase(
    model, train_loader, val_loader, criterion, opt_p2, sch_p2,
    PHASE2_EPOCHS, best_path,
    'PHASE 2 -- Fine-Tuning (block4 and block5 unfrozen)',
    history=history  # Extend the existing Phase 1 history
)

# Persist full training history for the comparison notebook
with open(RES_DIR / f'{MODEL_NAME}_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved to results/{MODEL_NAME}_history.json')

## Section 7: Learning Curves

In [ ]:
epochs_trained = len(history['train_loss'])
epochs_ran     = range(1, epochs_trained + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{MODEL_NAME} — Training and Validation Curves', fontsize=13, fontweight='bold')

for ax, tr_key, va_key, ylabel in [
    (axes[0], 'train_loss', 'val_loss', 'Cross-Entropy Loss'),
    (axes[1], 'train_acc',  'val_acc',  'Accuracy (%)'),
]:
    ytr = history[tr_key]
    yva = history[va_key]
    if 'acc' in tr_key:
        ytr = [v * 100 for v in ytr]
        yva = [v * 100 for v in yva]
    ax.plot(epochs_ran, ytr, 'b-', linewidth=1.5, label='Training')
    ax.plot(epochs_ran, yva, 'r-', linewidth=1.5, label='Validation', alpha=0.85)
    # Mark the Phase 2 start with a vertical line
    if epochs_trained > PHASE1_EPOCHS:
        ax.axvline(PHASE1_EPOCHS, color='orange', linestyle='--', alpha=0.8,
                   linewidth=1.5, label=f'Fine-tuning start (epoch {PHASE1_EPOCHS})')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel.split(" ")[0]} Curves')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_learning_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 8: Test Set Evaluation

In [ ]:
# Load the best checkpoint for final evaluation
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        logits = model(X_b.to(DEVICE))
        all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y_b.numpy())

all_preds = np.array(all_preds); all_labels = np.array(all_labels); all_probs = np.array(all_probs)

test_acc         = accuracy_score(all_labels, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
roc_auc          = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
total_p          = sum(p.numel() for p in model.parameters())
per_class_acc    = [accuracy_score(all_labels[all_labels==i], all_preds[all_labels==i])
                    for i in range(NUM_CLASSES)]

print('=' * 55)
print(f'  {MODEL_NAME} — Final Test Set Results')
print('=' * 55)
print(f'  Accuracy  (weighted) : {test_acc*100:.2f}%')
print(f'  Precision (weighted) : {prec*100:.2f}%')
print(f'  Recall    (weighted) : {rec*100:.2f}%')
print(f'  F1-Score  (weighted) : {f1*100:.2f}%')
print(f'  ROC-AUC   (OvR,wtd) : {roc_auc:.4f}')
print(f'  Total parameters     : {total_p:,}')
print('=' * 55)
print('\nPer-Class Classification Report:')
print(classification_report(all_labels, all_preds,
      target_names=[c.replace('_',' ').title() for c in CLASS_NAMES], digits=4))

## Section 9: Confusion Matrix and ROC Curves

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
pretty  = [c.replace('_', ' ').title() for c in CLASS_NAMES]

# --- Confusion matrix (absolute and normalised) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'{MODEL_NAME} — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
for ax, data, fmt, title, vmax in [
    (axes[0], cm,      'd',   'Absolute Counts', None),
    (axes[1], cm_norm, '.2f', 'Normalised',      1.0),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=pretty, yticklabels=pretty,
                linewidths=0.5, vmin=0, vmax=vmax,
                annot_kws={'fontsize': 10, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label'); ax.set_title(title)
    ax.tick_params(axis='x', rotation=25); ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

# --- ROC curves (one-vs-rest per class) ---
y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(9, 7))
for idx, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, idx], all_probs[:, idx])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{cls.replace("_"," ").title()} (AUC = {auc(fpr,tpr):.3f})')
ax.plot([0,1], [0,1], 'k--', linewidth=1.5, label='Random classifier (AUC = 0.500)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'{MODEL_NAME} — ROC Curves | Weighted AUC = {roc_auc:.4f}', fontsize=11)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

# --- Per-class accuracy bar chart ---
per_class_acc_pct = [a*100 for a in per_class_acc]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(pretty, per_class_acc_pct, color=CLASS_COLORS, alpha=0.85, edgecolor='white')
ax.axhline(test_acc*100, color='black', linestyle='--', linewidth=1.5,
           label=f'Overall accuracy ({test_acc*100:.1f}%)')
for bar, val in zip(bars, per_class_acc_pct):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 115); ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{MODEL_NAME} — Per-Class Test Accuracy', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_per_class_accuracy.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 10: Save Metrics

In [ ]:
report_dict = classification_report(all_labels, all_preds,
                                    target_names=CLASS_NAMES, output_dict=True)
metrics = {
    'model_name':         MODEL_NAME,
    'architecture_type':  'VGG16 Transfer Learning (ImageNet pretrained)',
    'test_accuracy':      float(test_acc),
    'weighted_precision': float(prec),
    'weighted_recall':    float(rec),
    'weighted_f1':        float(f1),
    'roc_auc_weighted':   float(roc_auc),
    'per_class_accuracy': {CLASS_NAMES[i]: float(per_class_acc[i]) for i in range(NUM_CLASSES)},
    'per_class_report':   report_dict,
    'total_params':       total_p,
    'epochs_trained':     epochs_trained,
    'fine_tuning':        True,
    'pretrained':         True,
    'pretrained_on':      'ImageNet-1K',
    'unfreeze_strategy':  'block4 and block5 (layers >= index 17)',
    'hyperparameters': {
        'batch_size':   BATCH_SIZE,
        'lr_phase1':    LR_PHASE1,
        'lr_phase2':    LR_PHASE2,
        'phase1_epochs': PHASE1_EPOCHS,
        'normalization': 'VGG16 ImageNet (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])',
    },
}
with open(RES_DIR / f'{MODEL_NAME}_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved to results/{MODEL_NAME}_metrics.json')
print(f'Summary: Accuracy={test_acc*100:.2f}% | F1={f1*100:.2f}% | AUC={roc_auc:.4f} | Params={total_p:,}')